# 第 3 章：动态规划 —— 当你"知道一切"时

Ch02 结尾我们用矩阵求逆一步解出了 V^π——但那是因为 5×5 网格只有 25 个状态。围棋有 10^170 个状态，矩阵求逆？连矩阵都存不下。**好在贝尔曼方程本身就是递归的**——递归的东西可以迭代着算：从一个随机的猜测出发，反复代入方程，直到不再变化。这就是动态规划（DP）。

先说清楚一个前提：DP 假设你**知道一切**（完整的 P 和 R）。这在真实世界几乎从不成立——所以有读者会问"学它干嘛"。两个理由：第一，它给后面所有近似算法提供了**标准答案**——你写的 TD/DQN/PPO 学得对不对，先跟 DP 的解对表；第二，DP 的两个核心操作（策略评估 + 贪心改进）会被后续算法原样继承，只是"精确"换成"采样"、"表格"换成"网络"。

> 🌍 **真实世界**：DP 并没有退场——电梯调度、库存管理、无人机路径规划这些**模型已知**的运筹问题今天仍在用 DP 求解；AlphaGo 的 MCTS 里也藏着 DP 的影子。

## 学习目标

1. 用**迭代策略评估** 解 $V^\pi$（不再依赖矩阵求逆）
2. 掌握**策略改进定理**（含完整证明）
3. 实现完整的 **策略迭代（Policy Iteration）**
4. 推导**贝尔曼最优性方程** 并实现**值迭代（Value Iteration）**
5. 看动画理解"扫描"过程

In [ ]:
import sys, pathlib
ROOT = pathlib.Path.cwd()
while not (ROOT / 'rlenvs').exists() and ROOT.parent != ROOT:
    ROOT = ROOT.parent
if str(ROOT) not in sys.path:
    sys.path.insert(0, str(ROOT))

import numpy as np
import matplotlib.pyplot as plt
from matplotlib import animation
from IPython.display import HTML
from utils import set_seed, plot_value_heatmap, make_interactive
from rlenvs import GridWorld, small_grid_5x5

set_seed(0)

## 3.1 DP 的假设：完美模型

动态规划（Dynamic Programming, DP）假设你**完全知道 MDP 的所有要素**：
- 转移概率 $P(s'|s,a)$
- 奖励函数 $R(s,a)$
- 状态集 $\mathcal{S}$、动作集 $\mathcal{A}$

这在现实中通常**不成立**——但你下棋知道规则、迷宫知道布局就是 DP 能直接处理的场景。
更重要的：**DP 是所有 model-free RL 算法（Ch04+）的理论基础**。

DP 的两大用途：

1. **预测（Prediction）**：给定 $\pi$，求 $V^\pi$
2. **控制（Control）**：求最优 $\pi^*$ 和 $V^*$

## 3.2 迭代策略评估（Iterative Policy Evaluation）

**问题**：给定 $\pi$，求 $V^\pi$。

Ch02 用了矩阵求逆 $V^\pi = (I - \gamma P^\pi)^{-1} R^\pi$，但矩阵求逆在状态空间大时（$10^5$ 状态以上）不可行。

**迭代法**：用一个近似序列 $V_0, V_1, V_2, \dots$ 收敛到 $V^\pi$。初始 $V_0$ 任意，每一步做"贝尔曼 backup"：

$$
V_{k+1}(s) = \sum_a \pi(a|s) \sum_{s'} P(s'|s,a) \big[ r(s,a,s') + \gamma V_k(s') \big]
$$

**直觉**：每一步把"当前对未来的估计"代回贝尔曼方程，得到更好的估计。

### 收敛性

可以证明（Ch02 矩阵形式 $V^\pi = R^\pi + \gamma P^\pi V^\pi$），$V_k \to V^\pi$ 当 $\gamma < 1$，收敛速率 $O(\gamma^k)$。

In [ ]:
def iterative_policy_eval(env, pi, gamma=0.9, theta=1e-6, max_iters=10000, record_history=False):
    """迭代策略评估。
    pi : [nS, nA] 的概率矩阵
    theta : 收敛阈值（最大变化 < theta 时停止）
    """
    nS, nA = env.nS, env.nA
    V = np.zeros(nS)
    history = [V.copy()] if record_history else None
    for it in range(max_iters):
        delta = 0.0
        new_V = np.zeros(nS)
        for s in range(nS):
            if env.is_terminal(s):
                continue
            v_old = V[s]
            v_new = 0.0
            for a in range(nA):
                if pi[s, a] == 0:
                    continue
                # Σ_{s'} P(s'|s,a) [r + γ V(s')]
                for s2 in range(nS):
                    p = env.P[s, a, s2]
                    if p == 0:
                        continue
                    # 注意：我们环境的 R 是 R[s, a] 形式，不依赖 s'
                    v_new += pi[s, a] * p * (env.R[s, a] + gamma * V[s2])
            new_V[s] = v_new
            delta = max(delta, abs(v_new - v_old))
        V = new_V
        if record_history:
            history.append(V.copy())
        if delta < theta:
            break
    return V, it + 1, history


env = small_grid_5x5(seed=0)
nS, nA = env.nS, env.nA
pi_uniform = np.full((nS, nA), 1.0 / nA)

V_dp, n_iters, history = iterative_policy_eval(
    env, pi_uniform, gamma=0.9, theta=1e-6, record_history=True
)
print(f"迭代策略评估在 {n_iters} 步收敛")

# 用矩阵法对比
pi = pi_uniform
R_pi = (pi * env.R).sum(axis=1)
P_pi = np.einsum('sa,saq->sq', pi, env.P)
V_exact = np.linalg.solve(np.eye(nS) - 0.9 * P_pi, R_pi)
print(f"DP vs 矩阵法 最大误差: {np.abs(V_dp - V_exact).max():.2e}")

### 向量化版本（快很多）

上面那个三重循环慢且啰嗦。用 numpy einsum 一下：

In [ ]:
def iterative_policy_eval_vec(env, pi, gamma=0.9, theta=1e-7, max_iters=10000, record=False):
    """向量化版本，比循环快 100 倍。"""
    nS, nA = env.nS, env.nA
    R_sa = env.R  # [nS, nA]
    P_sas = env.P  # [nS, nA, nS]
    V = np.zeros(nS)
    history = [V.copy()] if record else None
    for it in range(max_iters):
        # Q_k[s, a] = Σ_{s'} P(s'|s,a) [R(s,a) + γ V_k(s')]
        Q = R_sa + gamma * np.einsum('saq,q->sa', P_sas, V)
        # 注意：上式 R[s,a] 已经平均了 s'，但我们的 env 实现里 R[s,a] 不依赖 s'，直接用
        new_V = (pi * Q).sum(axis=1)
        # 终止态强制为 0
        for s in range(nS):
            if env.is_terminal(s):
                new_V[s] = 0.0
        delta = np.abs(new_V - V).max()
        V = new_V
        if record:
            history.append(V.copy())
        if delta < theta:
            break
    return V, it + 1, history


V_dp2, n_iters2, history2 = iterative_policy_eval_vec(env, pi_uniform, gamma=0.9, record=True)
print(f"向量化版本：{n_iters2} 步收敛，误差 {np.abs(V_dp2 - V_exact).max():.2e}")

## 3.3 策略改进：从 $V^\pi$ 找到更好的 $\pi'$

给定 $V^\pi$，我们能找一个更好的策略 $\pi'$ 吗？

### 贪心策略

$$
\pi'(s) = \arg\max_a \sum_{s'} P(s'|s,a) \big[ r(s,a,s') + \gamma V^\pi(s') \big]
$$

也就是"看哪个动作的短期奖励 + 长期价值最大"。这叫**对 $V^\pi$ 贪心**。

### 策略改进定理（核心！）

**定理**：设 $\pi'$ 是对 $V^\pi$ 贪心的策略。则对所有 $s$：

$$
V^{\pi'}(s) \geq V^\pi(s)
$$

<details>
<summary><b>📝 完整证明（点开看）</b></summary>

**目标**：证 $V^{\pi'}(s) \geq V^\pi(s), \forall s$。

记 $q_\pi(s, a) = \sum_{s'} P(s'|s,a)[r + \gamma V^\pi(s')]$。

**Step 1**：因为 $\pi'$ 对 $V^\pi$ 贪心，所以

$$
V^\pi(s) \leq q_\pi(s, \pi'(s)) = \sum_{s'} P(s'|s,\pi'(s)) [r + \gamma V^\pi(s')]
$$

**Step 2**：把右边那个 $V^\pi(s')$ 用同样办法展开（递归）

$$
\begin{aligned}
V^\pi(s) &\leq \mathbb{E}_{s' \sim P, \cdot | s, \pi'(s)} [r + \gamma V^\pi(s')] \\
         &\leq \mathbb{E}_{\pi'} [r_1 + \gamma r_2 + \gamma^2 V^\pi(s_2)] \quad \text{(再用一次贪心)} \\
         &\leq \mathbb{E}_{\pi'} \left[ \sum_{k=0}^{T} \gamma^k r_{k+1} + \gamma^{T+1} V^\pi(s_{T+1}) \right]
\end{aligned}
$$

**Step 3**：令 $T \to \infty$，$\gamma^{T+1} V^\pi(s_{T+1}) \to 0$（$\gamma < 1$ 保证），右边变成 $V^{\pi'}(s)$。所以

$$
V^\pi(s) \leq V^{\pi'}(s) \quad \blacksquare
$$

</details>

**意义**：贪心改进**永远不变差**。这是 RL 算法设计的一个基石。

In [ ]:
def greedy_policy_from_V(env, V, gamma=0.9):
    """对 V 贪心，返回确定性策略 [nS, nA]（one-hot）。"""
    nS, nA = env.nS, env.nA
    Q = np.zeros((nS, nA))
    for s in range(nS):
        if env.is_terminal(s):
            continue
        for a in range(nA):
            for s2 in range(nS):
                p = env.P[s, a, s2]
                if p > 0:
                    Q[s, a] += p * (env.R[s, a] + gamma * V[s2])
    pi = np.zeros((nS, nA))
    pi[np.arange(nS), Q.argmax(axis=1)] = 1.0
    # 终止态任选（反正不动作）
    return pi, Q


# 用 V_uniform 贪心，看新策略是否更好
pi_greedy, Q_greedy = greedy_policy_from_V(env, V_dp2, gamma=0.9)
V_greedy, _, _ = iterative_policy_eval_vec(env, pi_greedy, gamma=0.9)

print("改进后比改进前更优？", (V_greedy >= V_dp2 - 1e-6).all())
print(f"  V^π_uniform 平均: {V_dp2.mean():.3f}")
print(f"  V^π_greedy 平均: {V_greedy.mean():.3f}")

# 画两个 V 对比
fig, axes = plt.subplots(1, 2, figsize=(11, 5))
for ax, V_, title in [(axes[0], V_dp2, 'V (uniform)'), (axes[1], V_greedy, 'V (greedy improved)')]:
    plot_value_heatmap(
        V_, env.shape, cell_text=True,
        walls=list(env.walls), terminals=list(env.terminals),
        ax=ax, title=title,
    )
plt.tight_layout(); plt.show()

## 3.4 策略迭代（Policy Iteration）

把"评估"和"改进"交替进行：

```
初始化 π_0
repeat:
    V^π_k ← 迭代策略评估(π_k)
    π_{k+1} ← 对 V^π_k 贪心
until π_{k+1} == π_k
```

由策略改进定理，$V^{\pi_{k+1}} \geq V^{\pi_k}$，且当 $\pi$ 是最优时严格收敛。

In [ ]:
def policy_iteration(env, gamma=0.9, theta=1e-7, max_outer=50, record=False):
    nS, nA = env.nS, env.nA
    pi = np.full((nS, nA), 1.0 / nA)  # 初始均匀随机
    history_pi = [pi.copy()] if record else None
    history_V = [] if record else None
    for k in range(max_outer):
        V, _, _ = iterative_policy_eval_vec(env, pi, gamma=gamma, theta=theta)
        if record:
            history_V.append(V.copy())
        pi_new, _ = greedy_policy_from_V(env, V, gamma=gamma)
        if record:
            history_pi.append(pi_new.copy())
        if np.array_equal(pi_new.argmax(axis=1), pi.argmax(axis=1)):
            return V, pi, k + 1, history_V, history_pi
        pi = pi_new
    return V, pi, max_outer, history_V, history_pi


V_star_pi, pi_star, n_outer, hist_V, hist_pi = policy_iteration(env, gamma=0.9, record=True)
print(f"策略迭代在 {n_outer} 次外层迭代收敛")
print(f"V* 平均: {V_star_pi.mean():.3f}, max: {V_star_pi.max():.3f}")

# 画最优 V 和最优策略
arrows = ['↑','→','↓','←']
optimal_actions = pi_star.argmax(axis=1)
fig, ax = plt.subplots(figsize=(5, 5))
plot_value_heatmap(
    V_star_pi, env.shape, policy=optimal_actions, action_labels=arrows,
    cell_text=True, walls=list(env.walls), terminals=list(env.terminals),
    ax=ax, title='V* + π* (Policy Iteration)',
)
plt.tight_layout(); plt.show()

## 3.5 值迭代（Value Iteration）

策略迭代的问题：每次外层迭代都要做一次完整的策略评估（收敛到 $V^\pi$）——开销大。

**值迭代**的洞察：评估不必完全收敛！只要做一次"贝尔曼 backup"就够了：

$$
V_{k+1}(s) = \max_a \sum_{s'} P(s'|s,a) \big[ r(s,a,s') + \gamma V_k(s') \big]
$$

注意和迭代策略评估的区别：多了个 $\max_a$。这就是**贝尔曼最优性方程**的迭代形式。

### 贝尔曼最优性方程

最优价值函数 $V^*$ 满足：

$$
\boxed{\; V^*(s) = \max_a \sum_{s'} P(s'|s,a) \big[ r(s,a,s') + \gamma V^*(s') \big] \;}
$$

这是一个**非线性**方程（因为 max），所以不能直接矩阵求逆。但**值迭代**可以收敛到它。

### 策略迭代 vs 值迭代

| | 策略迭代 | 值迭代 |
|---|---|---|
| 外层 | 改进策略 | 改进 V |
| 内层 | 完整策略评估 | 一次 backup |
| 复杂度 | 每个外层多步内层 | 每步都是一次 backup |
| 实际效率 | 状态多时慢 | 通常更快 |

两者最终都收敛到 $V^*$。

In [ ]:
def value_iteration(env, gamma=0.9, theta=1e-7, max_iters=10000, record=False):
    """值迭代。"""
    nS, nA = env.nS, env.nA
    V = np.zeros(nS)
    history = [V.copy()] if record else None
    for it in range(max_iters):
        # Q[s, a] = Σ_{s'} P(s'|s,a) [R(s,a) + γ V(s')]
        Q = env.R + gamma * np.einsum('saq,q->sa', env.P, V)
        new_V = Q.max(axis=1)
        # 终止态为 0
        for s in range(nS):
            if env.is_terminal(s):
                new_V[s] = 0.0
        delta = np.abs(new_V - V).max()
        V = new_V
        if record:
            history.append(V.copy())
        if delta < theta:
            break
    # 提取最优策略
    Q_final = env.R + gamma * np.einsum('saq,q->sa', env.P, V)
    pi = np.zeros((nS, nA))
    pi[np.arange(nS), Q_final.argmax(axis=1)] = 1.0
    return V, pi, it + 1, history


V_star_vi, pi_star_vi, n_iters_vi, hist_V_vi = value_iteration(env, gamma=0.9, record=True)
print(f"值迭代在 {n_iters_vi} 次迭代收敛")
print(f"V* 与策略迭代结果的差异: {np.abs(V_star_vi - V_star_pi).max():.2e}")

## 3.6 值迭代动画：看 V 怎么收敛

> 🤔 **先猜再跑**：动画从 V₀ = 0（一片黑）开始播放值迭代的每一轮扫描。预测一下**第 1 轮扫描后**，网格里哪些格子已经有值、哪些还是 0？
>
> <details><summary>想好了再点开</summary>
>
> 第 1 轮只有**终点的直接邻居**非零：贝尔曼 backup 一次只能"看到一步远"的奖励。像水波从终点荡开，每扫一轮多传一格——这个"波速"就是后面 TD 学习（Ch04）里 credit 沿时间回传的速度上限。
> </details>

下面动画展示值迭代从 $V_0 = 0$ 开始的演化。你应该看到 **奖励信号从终点 +1 倒着传回起点**。

In [ ]:
# 因为值迭代收敛快（~30 步），我们直接画前 20 步
fig, ax = plt.subplots(figsize=(5, 5))
n_frames = min(20, len(hist_V_vi))

def update(k):
    ax.clear()
    plot_value_heatmap(
        hist_V_vi[k], env.shape,
        cell_text=True, walls=list(env.walls), terminals=list(env.terminals),
        ax=ax, title=f'value iteration  iter={k}',
    )

anim = animation.FuncAnimation(fig, update, frames=n_frames, interval=400, blit=False, repeat=True)
plt.close(fig)
HTML(anim.to_jshtml())

## 3.7 比较两个算法的效率

我们让两个算法都收敛到 $\theta = 10^{-7}$，看哪个快。

In [ ]:
import time

# 值迭代
t0 = time.time()
V_vi, pi_vi, n_vi, _ = value_iteration(env, gamma=0.9, theta=1e-7)
t_vi = time.time() - t0

# 策略迭代
t0 = time.time()
V_pi, pi_pi, n_pi_outer, _, _ = policy_iteration(env, gamma=0.9, theta=1e-7)
t_pi = time.time() - t0

print(f"{'算法':<14}{'总迭代数':<14}{'时间(s)':<10}{'最终 max V':<10}")
print(f"{'值迭代':<14}{n_vi:<14}{t_vi:<10.3f}{V_vi.max():<10.3f}")
print(f"{'策略迭代':<14}{n_pi_outer:<14}{t_pi:<10.3f}{V_pi.max():<10.3f}")
print(f"\n两者 V 的差异: {np.abs(V_vi - V_pi).max():.2e}")

## 3.8 数值验证：策略迭代的单调改进

策略改进定理说每步 $V^{\pi_{k+1}} \geq V^{\pi_k}$。我们看一下记录的 history 是不是这样。

In [ ]:
# 我们重跑一次，记录每个外层迭代的 V
print(f"{'iter':<6}{'mean(V^π_k)':<14}{'max(V^π_k)':<14}{'Δ':<12}")
prev_mean = -np.inf
for k, Vk in enumerate(hist_V):
    cur_mean = Vk.mean()
    delta = cur_mean - prev_mean if k > 0 else 0
    print(f"{k:<6}{cur_mean:<14.4f}{Vk.max():<14.4f}{delta:<+12.4f}")
    prev_mean = cur_mean

你应该看到 $V$ 的均值**单调递增**，且增量很快变小——这就是策略改进定理的数值体现。

## 3.9 📝 练习

### 练习 1：Modified Policy Iteration

策略迭代每次评估到收敛 $\theta = 10^{-7}$，开销大。一个折中方案叫 **Modified Policy Iteration**：

- 每次评估只做 $k$ 次 sweep（不等到收敛）
- 然后立即贪心改进

**任务**：
1. 实现 `modified_policy_iteration(env, k=5, ...)`
2. 对比 $k \in \{1, 2, 5, 10, 20, 50\}$，找出能"以最少总运算量收敛到接近最优"的 $k$
3. 画出 $k$ vs 总 sweep 数 + 最终 $V$ 误差

**预期结果**：$k \approx 5$ 通常比纯策略迭代（$k \to \infty$）和纯值迭代（$k=1$）都好。

> 参考答案：`solutions/ch03_modified_policy_iteration.ipynb`

---

## 3.10 小结

| 算法 | 每轮做什么 | 收敛到 |
|---|---|---|
| 策略迭代 | 完整评估 $V^\pi$ → 贪心改进 | $\pi^*$ |
| 值迭代 | 一次贝尔曼最优回扫 $V \leftarrow \max_a \dots$ | $V^*$ |
| Modified PI | $k$ 次 sweep 评估 + 改进（练习 1） | $\pi^*$ |

三个带走的东西：

1. **DP = 已知模型时的精确解法**：它并不实用（需要完整 $P$、状态数爆炸），但给后面所有近似算法提供了"标准答案"来对照
2. **策略改进定理**是迭代收敛的数学保证：贪心改进永不变差、有限策略数内必达最优
3. **γ<1 ⇒ 压缩映射**：误差每轮以 γ 几何衰减——这套收敛分析在 Ch04 的 TD 学习里会原样复用

> 📖 学完本章，先做 `STUDY_GUIDE.md` 里 Ch03 的自测题（3 题），全对再进下一章。

---

下一章：**第 4 章 — TD 学习**。
我们将放弃 "知道 P 和 R" 的假设——agent 必须**从交互样本中学习**。这是 RL 真正走入现实的起点。